# ID3 Decision Tree Implementation from Scratch

This notebook implements the ID3 Decision Tree algorithm from scratch, without relying on `scikit-learn` or any other machine learning libraries. It is designed to handle continuous features (like those found in the Iris dataset) by finding the optimal threshold for binary splits at each node.

## Objectives:
- Implement standard Entropy and Information Gain calculations.
- Build a recursive tree-building algorithm.
- Accept test examples and classify them correctly.
- Evaluate the performance of the implemented algorithm.

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Set random seed for reproducibility
np.random.seed(42)

## 1. Data Loading and Preprocessing
We load the `iris.csv` dataset, drop the 'ID' column as it is not a predictive feature, and split our data into training and testing sets (from scratch).

In [4]:
# Load dataset
df = pd.read_csv('iris.csv')
if 'ID' in df.columns:
    df = df.drop('ID', axis=1)

def train_test_split_scratch(df, test_size=0.2):
    # Shuffle the dataframe
    indices = np.random.permutation(df.index)
    test_set_size = int(len(df) * test_size)
    
    test_indices = indices[:test_set_size]
    train_indices = indices[test_set_size:]
    
    return df.iloc[train_indices], df.iloc[test_indices]

train_df, test_df = train_test_split_scratch(df, test_size=0.2)

X_train = train_df.drop('class', axis=1).values
y_train = train_df['class'].values

X_test = test_df.drop('class', axis=1).values
y_test = test_df['class'].values

print(f"Training set size: {len(X_train)}")
print(f"Testing set size: {len(X_test)}")

Training set size: 120
Testing set size: 30


## 2. Information Theory Metrics
The core of ID3 relies on Entropy to measure impurity, and Information Gain to determine the best feature (and threshold) to split on.

In [5]:
def entropy(y):
    """Calculates the entropy of a label array."""
    classes, counts = np.unique(y, return_counts=True)
    probabilities = counts / len(y)
    return -np.sum(probabilities * np.log2(probabilities + 1e-9)) # added epsilon to prevent log(0)

def information_gain(y, y_left, y_right):
    """Calculates the Information Gain of a split."""
    p_left = len(y_left) / len(y)
    p_right = len(y_right) / len(y)
    return entropy(y) - (p_left * entropy(y_left) + p_right * entropy(y_right))

## 3. Decision Tree Implementation
We implement a `Node` class to hold tree structure information and an `ID3Tree` class that contains the recursive learning and predictive logic.

In [6]:
class Node:
    def __init__(self, feature=None, threshold=None, left=None, right=None, *, value=None):
        self.feature = feature       # Index of feature to split on
        self.threshold = threshold   # Threshold value for continuous feature
        self.left = left             # Left child (Node)
        self.right = right           # Right child (Node)
        self.value = value           # Class label if it's a leaf node
        
    def is_leaf(self):
        return self.value is not None

class ID3Tree:
    def __init__(self, min_samples_split=2, max_depth=100):
        self.min_samples_split = min_samples_split
        self.max_depth = max_depth
        self.root = None
        
    def fit(self, X, y):
        self.root = self._grow_tree(X, y, depth=0)
        
    def _grow_tree(self, X, y, depth):
        n_samples, n_features = X.shape
        n_labels = len(np.unique(y))
        
        # Stopping criteria
        if (depth >= self.max_depth or n_labels == 1 or n_samples < self.min_samples_split):
            leaf_value = self._most_common_label(y)
            return Node(value=leaf_value)
        
        # Find the best split
        best_feat, best_thresh = self._best_split(X, y, n_features)
        
        # If no valid split is found, return a leaf
        if best_feat is None:
            return Node(value=self._most_common_label(y))
            
        left_idxs, right_idxs = self._split(X[:, best_feat], best_thresh)
        
        # Recursive growth
        left_child = self._grow_tree(X[left_idxs, :], y[left_idxs], depth + 1)
        right_child = self._grow_tree(X[right_idxs, :], y[right_idxs], depth + 1)
        
        return Node(feature=best_feat, threshold=best_thresh, left=left_child, right=right_child)
    
    def _best_split(self, X, y, n_features):
        best_gain = -1
        split_idx, split_thresh = None, None
        
        for feat_idx in range(n_features):
            X_column = X[:, feat_idx]
            thresholds = np.unique(X_column)
            
            for thr in thresholds:
                left_idxs, right_idxs = self._split(X_column, thr)
                
                if len(left_idxs) == 0 or len(right_idxs) == 0:
                    continue
                    
                gain = information_gain(y, y[left_idxs], y[right_idxs])
                
                if gain > best_gain:
                    best_gain = gain
                    split_idx = feat_idx
                    split_thresh = thr
                    
        return split_idx, split_thresh
    
    def _split(self, X_column, split_thresh):
        left_idxs = np.argwhere(X_column <= split_thresh).flatten()
        right_idxs = np.argwhere(X_column > split_thresh).flatten()
        return left_idxs, right_idxs
    
    def _most_common_label(self, y):
        classes, counts = np.unique(y, return_counts=True)
        return classes[np.argmax(counts)]
    
    def predict(self, X):
        return np.array([self._traverse_tree(x, self.root) for x in X])
    
    def _traverse_tree(self, x, node):
        if node.is_leaf():
            return node.value
        
        if x[node.feature] <= node.threshold:
            return self._traverse_tree(x, node.left)
        return self._traverse_tree(x, node.right)

## 4. Model Training
We now instantiate our custom `ID3Tree` and fit it using the training data.

In [7]:
tree = ID3Tree(max_depth=5)
tree.fit(X_train, y_train)
print("Tree training completed!")

Tree training completed!


## 5. Model Evaluation and Prediction
We evaluate the model on the testing set and calculate accuracy. 
Additionally, we demonstrate how the tree handles entirely new examples.

In [8]:
def accuracy(y_true, y_pred):
    return np.sum(y_true == y_pred) / len(y_true)

y_pred = tree.predict(X_test)
acc = accuracy(y_test, y_pred)
print(f"Accuracy on test set: {acc * 100:.2f}%")

print("\n--- Sample Predictions vs Actual ---")
for i in range(5):
    print(f"Predicted: {y_pred[i]:<15} | Actual: {y_test[i]}")

Accuracy on test set: 100.00%

--- Sample Predictions vs Actual ---
Predicted: Iris-versicolor | Actual: Iris-versicolor
Predicted: Iris-setosa     | Actual: Iris-setosa
Predicted: Iris-virginica  | Actual: Iris-virginica
Predicted: Iris-versicolor | Actual: Iris-versicolor
Predicted: Iris-versicolor | Actual: Iris-versicolor


### Predicting New Examples
The assignment specifically asks to apply the generated tree to new examples. Let's create some hypothetical iris flowers and classify them.

In [9]:
# New unclassified examples (SepalLength, SepalWidth, PetalLength, PetalWidth)
new_examples = np.array([
    [5.0, 3.5, 1.5, 0.2],  # Expected: Iris-setosa
    [6.5, 3.0, 4.5, 1.5],  # Expected: Iris-versicolor
    [7.5, 3.8, 6.5, 2.2]   # Expected: Iris-virginica
])

new_predictions = tree.predict(new_examples)

print("Classifying new test examples:")
for i, example in enumerate(new_examples):
    print(f"Features: {example} -> Predicted Class: {new_predictions[i]}")

Classifying new test examples:
Features: [5.  3.5 1.5 0.2] -> Predicted Class: Iris-setosa
Features: [6.5 3.  4.5 1.5] -> Predicted Class: Iris-versicolor
Features: [7.5 3.8 6.5 2.2] -> Predicted Class: Iris-virginica
